In [0]:
%run ../init/kafka_config

In [0]:
from confluent_kafka import Consumer, Producer
from delta.tables import DeltaTable
from pyspark.sql.functions import col
from datetime import datetime
import json
import uuid

In [0]:
# ── Initialize Consumer ────────────────────
consumer = Consumer(consumer_conf)
consumer.subscribe(["orders"])

# ── Initialize Producer (for dead_letter & notifications) ──
producer = Producer(producer_conf)

def delivery_report(err, msg):
    if err:
        print(f"✗ Failed to send: {err}")

In [0]:
def get_user(user_id):
    """Returns user row or None"""
    df = spark.read.table("users").filter(col("user_id") == user_id)
    return df.first() if df.count() > 0 else None

def get_item(item_id):
    """Returns item row or None"""
    df = spark.read.table("items").filter(col("item_id") == item_id)
    return df.first() if df.count() > 0 else None

def get_inventory(item_id):
    """Returns inventory row or None"""
    df = spark.read.table("inventory").filter(col("item_id") == item_id)
    return df.first() if df.count() > 0 else None

def is_duplicate(order_id):
    """Check if order_id already exists in orders table"""
    df = spark.read.table("orders").filter(col("order_id") == order_id)
    return df.count() > 0


In [0]:
def send_to_dead_letter(order, reason):
    """Send failed order to dead_letter topic"""
    payload = {**order, "failure_reason": reason}
    producer.produce(
        topic="dead_letter",
        value=json.dumps(payload).encode("utf-8"),
        callback=delivery_report
    )
    producer.poll(0)
    print(f"✗ Order {order['order_id']} → dead_letter | reason: {reason}")

In [0]:
def write_order_log(order, status, balance_before):
    """Write to order_logs table"""
    log_data = [(
        order["order_id"],
        order["user_id"],
        order["item_id"],
        order["item_quantity"],
        status,
        balance_before,
        datetime.now()
    )]
    log_schema = spark.read.table("order_logs").schema
    spark.createDataFrame(log_data, log_schema) \
        .write.format("delta") \
        .mode("append") \
        .saveAsTable("order_logs")

In [0]:
def write_inventory_log(item_id, qty_before, qty_after):
    """Write to inventory_logs table"""
    log_data = [(item_id, qty_before, qty_after, datetime.now())]
    log_schema = spark.read.table("inventory_logs").schema
    spark.createDataFrame(log_data, log_schema) \
        .write.format("delta") \
        .mode("append") \
        .saveAsTable("inventory_logs")

In [0]:
def write_notification(user, item, order, balance_after):
    """Write to notifications table"""
    notif_data = [(
        user["user_id"],
        user["username"],
        item["item_name"],
        order["item_quantity"],
        balance_after,
        datetime.now()
    )]
    notif_schema = spark.read.table("notifications").schema
    spark.createDataFrame(notif_data, notif_schema) \
        .write.format("delta") \
        .mode("append") \
        .saveAsTable("notifications")


In [0]:
def process_order(order):
    print(f"\nProcessing order: {order['order_id']}")

    # ── Step 1: Duplicate check ────────────
    if is_duplicate(order["order_id"]):
        send_to_dead_letter(order, "FAILED_DUPLICATE")
        return

    # ── Step 2: Validate user ──────────────
    user = get_user(order["user_id"])
    if user is None:
        send_to_dead_letter(order, "FAILED_INVALID_USER")
        write_order_log(order, "FAILED_INVALID_USER", 0.0)
        return

    # ── Step 3: Validate item ──────────────
    item = get_item(order["item_id"])
    if item is None:
        send_to_dead_letter(order, "FAILED_INVALID_ITEM")
        write_order_log(order, "FAILED_INVALID_ITEM", user["balance"])
        return

    # ── Step 4: Check inventory ────────────
    inventory = get_inventory(order["item_id"])
    if inventory is None or inventory["quantity_available"] < order["item_quantity"]:
        send_to_dead_letter(order, "FAILED_INVENTORY")
        write_order_log(order, "FAILED_INVENTORY", user["balance"])
        return

    # ── Step 5: Check balance ──────────────
    total_cost = item["price"] * order["item_quantity"]
    balance_before = user["balance"]

    if balance_before < total_cost:
        send_to_dead_letter(order, "FAILED_BALANCE")
        write_order_log(order, "FAILED_BALANCE", balance_before)
        return

    # ── All checks passed ──────────────────
    balance_after   = balance_before - total_cost
    qty_before      = inventory["quantity_available"]
    qty_after       = qty_before - order["item_quantity"]

    # ── Step 6: Deduct user balance ────────
    DeltaTable.forName(spark, "users").update(
        condition = col("user_id") == order["user_id"],
        set       = {"balance": str(balance_after)}
    )

    # ── Step 7: Deduct inventory ───────────
    DeltaTable.forName(spark, "inventory").update(
        condition = col("item_id") == order["item_id"],
        set       = {"quantity_available": str(qty_after)}
    )

    # ── Step 8: Write order record ─────────
    order_data = [(
        order["order_id"],
        order["user_id"],
        order["item_id"],
        order["item_quantity"],
        order["category"],
        "ORDER_PLACED",
        datetime.now()
    )]
    order_schema = spark.read.table("orders").schema
    spark.createDataFrame(order_data, order_schema) \
        .write.format("delta") \
        .mode("append") \
        .saveAsTable("orders")

    # ── Step 9: Write order log ────────────
    write_order_log(order, "ORDER_PLACED", balance_before)

    # ── Step 10: Write inventory log ───────
    write_inventory_log(order["item_id"], qty_before, qty_after)

    # ── Step 11: Write notification ────────
    write_notification(user, item, order, balance_after)

    print(f"✓ Order {order['order_id']} placed successfully | cost: {total_cost} | balance left: {balance_after}")


In [0]:
print("Starting order processor...\n")
processed = 0
failed    = 0

for _ in range(30):  # poll 30 times, covers all pending messages
    msg = consumer.poll(timeout=2.0)

    if msg is None:
        continue
    if msg.error():
        print(f"Consumer error: {msg.error()}")
        continue

    order = json.loads(msg.value().decode("utf-8"))

    try:
        process_order(order)
        processed += 1
    except Exception as e:
        print(f"✗ Unexpected error for order {order.get('order_id')}: {e}")
        send_to_dead_letter(order, f"FAILED_UNEXPECTED: {str(e)}")
        failed += 1

consumer.close()
producer.flush()

print(f"\n── Summary ─────────────────────────────")
print(f"Total processed: {processed}")
print(f"Total failed:    {failed}")
print(f"✓ Order processor complete")

In [0]:
print("── Orders ──────────────────────────────")
spark.read.table("orders").show(truncate=False)

print("── Order Logs ──────────────────────────")
spark.read.table("order_logs").show(truncate=False)

print("── Inventory Logs ──────────────────────")
spark.read.table("inventory_logs").show(truncate=False)

print("── Notifications ───────────────────────")
spark.read.table("notifications").show(truncate=False)

print("── Users (balance updated) ─────────────")
spark.read.table("users").show(truncate=False)

print("── Inventory (stock updated) ───────────")
spark.read.table("inventory").show(truncate=False)